# Quantum Information Theory - General Formulas

In [408]:
import math
import statistics
import numpy as np
from itertools import permutations

## From Tests

### Simple List Product - For Probability Computation

In [409]:
p = [4/24, 6/24]
print(math.prod(p))

0.041666666666666664


### Expected Value $E[X]$
$$E[X] = \sum_i x_i \cdot p_i$$

In [410]:
x = [3/24, 1/24, 4/24, 6/24, 2/24, 8/24]
p = [6/24, 2/24, 8/24, 1/24, 4/24, 3/24]
print(sum(xi * pi for xi, pi in zip(x, p)))

0.15625


#### $E[X^2]$

$$E[X^2] = \sum_i x_i^2 \cdot p_i$$


In [411]:
x = [3/24, 1/24, 4/24, 6/24, 2/24, 8/24]
p = [6/24, 2/24, 8/24, 1/24, 4/24, 3/24]
print(sum(xi * xi * pi for xi, pi in zip(x, p)))

0.030960648148148147


#### Variance $Var[X]$
$$Var[X] = E[X^2] - (E[X])^2$$

In [412]:
x = [3/24, 1/24, 4/24, 6/24, 2/24, 8/24]
p = [6/24, 2/24, 8/24, 1/24, 4/24, 3/24]
ex = sum(xi * pi for xi, pi in zip(x, p))
ex2 = sum(xi * xi * pi for xi, pi in zip(x, p))
print(ex2 - ex * ex)

0.006546585648148147


#### Standard deviation $\sigma_X$ or $std[X]$

$$\sigma_X = \sqrt{Var[X]}$$

In [413]:
x = [3/24, 1/24, 4/24, 6/24, 2/24, 8/24]
p = [6/24, 2/24, 8/24, 1/24, 4/24, 3/24]
ex = sum(xi * pi for xi, pi in zip(x, p))
ex2 = sum(xi * xi * pi for xi, pi in zip(x, p))
print(math.sqrt(ex2 - ex * ex))

0.08091097359535446


### Median winner - Median value

In [414]:
p = [0.03, 0.12, 0.36, 0.15, 0.24, 0.1]
print(statistics.median(p))

0.135


### Odds Problems

#### Simple take-in bets %
Set the odds such that on average we keep 10% of the take-in-bets, if we expect a balanced book.

In [415]:
p = [0.03, 0.12, 0.36, 0.15, 0.24, 0.1]
take_in = 0.1
print([str((1-take_in)/x) + "x" for x in p])

['30.000000000000004x', '7.500000000000001x', '2.5x', '6.0x', '3.7500000000000004x', '9.0x']


#### Betting on the underdog
Set the odds such that on average we keep 10% if we expect the underdog to be bet on 4x as much as any other horse.

In [416]:
p = [0.03, 0.12, 0.36, 0.15, 0.24, 0.1]
take_in = 0.1
underdog_idx = p.index(min(p))  # A = underdog (lowest probability)
k = 4  # underdog bet k times more
n = len(p)

# Betting weights: underdog gets k, others get 1
bets = [1] * n
bets[underdog_idx] = k
T = sum(bets)

# For expected 10% margin: sum(b_i/T * p_i * o_i) = 1 - take_in
# With o_i = (1-take_in)/p_i this always holds, but doesn't account for risk.
# The corrected odds reduce underdog payout and increase others:
# o_i = c / p_i, where c is chosen per group so weighted avg return = 0.9
# Using structure: o_underdog = c_u/p_u, o_others = c_o/p_i
# Constraint: (k * c_u + (n-1) * c_o) / T = 1 - take_in

# Simple balanced-book correction:
odds = [(1 - take_in) * T / bets[i] for i in range(n)]
print("Balanced book odds:", [f"{o:.4f}x" for o in odds])

# Expected-value odds (keeping o_i proportional to 1/p_i):
# Standard: o_i = 0.9/p_i for all
print("Standard odds:    ", [str(round((1-take_in)/x, 4)) + "x" for x in p])

Balanced book odds: ['2.0250x', '8.1000x', '8.1000x', '8.1000x', '8.1000x', '8.1000x']
Standard odds:     ['30.0x', '7.5x', '2.5x', '6.0x', '3.75x', '9.0x']


#### Setting multiple bets
We set odds to keep 10% of the take-in for a balanced book. The underdog was bet on 4x as much as any other horse. We now expect 3x additional bets, with this distribution, and want to correct the odds to keep 10%.

In [417]:
p = [0.03, 0.12, 0.36, 0.15, 0.24, 0.1]
take_in = 0.1
k = 4  # underdog bet kx more
additional = 3  # 3x additional bets with same distribution

# Phase 1: [4,1,1,1,1,1], Phase 2: 3x same = [12,3,3,3,3,3]
# Total bets: [16,4,4,4,4,4], T = 36
underdog_idx = p.index(min(p))
n = len(p)
bets_per_round = [1] * n
bets_per_round[underdog_idx] = k
total_bets = [(1 + additional) * b for b in bets_per_round]
T = sum(total_bets)

# Check which answer set gives 10% expected margin:
# E[payout] = sum(p_i * b_i * o_i) = (1-take_in) * T
answers = {
    "Red":    [5, 12.5, 25/6, 10, 6.25, 15],
    "Blue":   [1.875, 13.125, 4.375, 10.5, 6.5625, 15.75],
    "Yellow": [10/3, 65/6, 65/18, 26/3, 65/12, 13],
    "Green":  [0, 11.25, 3.75, 9, 5.625, 13.5],
}

target = (1 - take_in) * T
print(f"Total bets: {total_bets}, T = {T}, target E[payout] = {target}\n")
for name, odds in answers.items():
    ep = sum(pi * bi * oi for pi, bi, oi in zip(p, total_bets, odds))
    margin = 1 - ep / T
    print(f"{name}: E[payout]={ep:.2f}, margin={margin:.2%} {'✓' if abs(margin-0.1)<0.001 else '✗'}")

Total bets: [16, 4, 4, 4, 4, 4], T = 36, target E[payout] = 32.4

Red: E[payout]=32.40, margin=10.00% ✓
Blue: E[payout]=32.40, margin=10.00% ✓
Yellow: E[payout]=27.60, margin=23.33% ✗
Green: E[payout]=27.00, margin=25.00% ✗


## From midterm review
We are streaming (and taking bets) for virtual 6-equine races. The winner is distributed as follows:

$$W \sim \begin{pmatrix} A & B & C & D & E & F \\ \frac{16}{61} & \frac{11}{61} & \frac{10}{61} & \frac{10}{61} & \frac{6}{61} & \frac{8}{61} \end{pmatrix}.$$


#### Problem 1
The following odds let us on average keep 10% of the take-in money in the case of a balanced-book:

   **A)** 3.507, 5.102, 5.612, 5.612, 9.353, 7.015 &emsp; **B)** 3.531, 4.891, 5.49, 5.49, 9.05, 6.962

   **C)** 3.431, 4.991, 5.49, 5.49, 9.15, 6.862 &emsp; **D)** 3.342, 5.225, 5.49, 5.49, 10.07, 6.447
   Answer: C

In [418]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
take_in = 0.1
print([str((1-take_in)/x) + "x" for x in p])

['3.43125x', '4.990909090909091x', '5.49x', '5.49x', '9.15x', '6.8625x']


#### Problem 2
The following odds let us on average keep 10% of the take-in money if we expect patterned competitors to be bet on twice as much as non-patterned ones:

   **A)** 4.708, 6.521, 3.66, 3.66, 12.07, 9.283 &emsp; **B)** 4.456, 6.967, 3.66, 3.66, 13.43, 8.596

   **C)** 4.575, 6.655, 3.66, 3.66, 12.2, 9.15 &emsp; **D)** 4.677, 6.802, 3.741, 3.741, 12.47, 9.353
   Answer: C

In [419]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
take_in = 0.1
k = 2  # patterned bet 2x more, not 4x

# Patterned = C and D (indices 2, 3) — bet kx more
bets = [1] * len(p)
bets[2] = k
bets[3] = k
T = sum(bets)

# Balanced book odds: o_i = (1-take_in) * T / b_i
odds = [(1 - take_in) * T / bets[i] for i in range(len(p))]
print("Balanced book odds:", [f"{o:.3f}x" for o in odds])

# Standard odds (no betting adjustment): o_i = 0.9/p_i
print("Standard odds:    ", [f"{(1-take_in)/x:.3f}x" for x in p])

# Check each answer
answers = {
    "A": [4.708, 6.521, 3.66, 3.66, 12.07, 9.283],
    "B": [4.456, 6.967, 3.66, 3.66, 13.43, 8.596],
    "C": [4.575, 6.655, 3.66, 3.66, 12.2, 9.15],
    "D": [4.677, 6.802, 3.741, 3.741, 12.47, 9.353],
}

target = (1 - take_in) * T
print(f"\nBets: {bets}, T = {T}, target E[payout] = {target}\n")
for name, o in answers.items():
    ep = sum(pi * bi * oi for pi, bi, oi in zip(p, bets, o))
    margin = 1 - ep / T
    print(f"{name}: E[payout]={ep:.4f}, margin={margin:.4%} {'✓' if abs(margin-0.1)<0.001 else '✗'}")

Balanced book odds: ['7.200x', '7.200x', '3.600x', '3.600x', '7.200x', '7.200x']
Standard odds:     ['3.431x', '4.991x', '5.490x', '5.490x', '9.150x', '6.862x']

Bets: [1, 1, 2, 2, 1, 1], T = 8, target E[payout] = 7.2

A: E[payout]=7.2155, margin=9.8068% ✗
B: E[payout]=7.2735, margin=9.0818% ✗
C: E[payout]=7.2001, margin=9.9990% ✓
D: E[payout]=7.3596, margin=8.0045% ✗


#### Problem 3 - Shannon Entropy $H(X)$
Streaming the sequence of winners, how many bits do we need on average per race?

   **A)** 1.747 &emsp; **B)** 2.458 &emsp; **C)** 2.049 &emsp; **D)** 2.521
   Answer: D

In [420]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
H = -sum(pi * math.log2(pi) for pi in p if pi > 0)
print(f'H(X) = {H:.3f} bits')

H(X) = 2.521 bits


#### Problem 4 - Horse???
If we only sent whether the winner was a horse, how many bits per race do we need then?

   **A)** 0.9951 &emsp; **B)** 1 &emsp; **C)** 0.541 &emsp; **D)** 0.5
Answer: A - just created a new probability distribution from the old one

In [421]:
p_all = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
# Horses are 1, 2, 5 (indices 0, 1, 4)
p_horse = p_all[0] + p_all[1] + p_all[4]  # 33/61
p_not = 1 - p_horse                         # 28/61

# Binary entropy: H(horse or not)
p = [p_horse, p_not]
H = -sum(pi * math.log2(pi) for pi in p if pi > 0)
print(f'P(horse) = {p_horse:.4f}, P(not) = {p_not:.4f}')
print(f'H = {H:.4f} bits')

P(horse) = 0.5410, P(not) = 0.4590
H = 0.9951 bits


#### Problem 5 - Quessing Entropy $E[G]$
Let's say we want to guess the winner. How many guesses do we need to make on average?

   **A)** 3.984 &emsp; **B)** 2.016 &emsp; **C)** 3.016 &emsp; **D)** 2.984
Answer: C

In [422]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
p_sorted = sorted(p, reverse=True)
EG = sum(i * pi for i, pi in enumerate(p_sorted, 1))
print(f'E[G] = {EG:.4f}')

E[G] = 3.0164


#### Problem 6 - Gini coefficient
The number of guesses required is a numeric variable. What is its Gini coefficient?

   **A)** 0.1612 &emsp; **B)** 0.4683 &emsp; **C)** 0.4732 &emsp; **D)** 0.3131
Answer: A

In [423]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
n = len(p)
# E[G] = guessing entropy (sorted descending, i starts from 1)
p_sorted = sorted(p, reverse=True)
EG = sum(i * pi for i, pi in enumerate(p_sorted, 1))
gini = (n + 1 - 2 * EG) / n
print(f'E[G] = {EG:.4f}')
print(f'Gini = {gini:.4f}')

E[G] = 3.0164
Gini = 0.1612


#### Problem 7
From now on, we double the win probabilities of the competitors with glasses then renormalize.

What is the new distribution of the winners?

   **A)** 0.4156, 0.1429, 0.1299, 0.1299, 0.07792, 0.1039 &emsp; **B)** 0.387, 0.133, 0.1209, 0.1209, 0.07256, 0.09675

   **C)** 0.4496, 0.1546, 0.1405, 0.1405, 0.08431, 0.1124 &emsp; **D)** 0.5246, 0.1803, 0.1639, 0.1639, 0.09836, 0.1311
   Answer A

In [424]:
p = [16/61 *2, 11/61, 10/61, 10/61, 6/61, 8/61]
p_prime = [x / sum(p) for x in p]
print(p)

[0.5245901639344263, 0.18032786885245902, 0.16393442622950818, 0.16393442622950818, 0.09836065573770492, 0.13114754098360656]


#### Problem 8 - Majorization
Does the new probabilistic vector $\mathbf{p}'$ majorize the old one $\mathbf{p}$?

   **A)** *neither* &emsp; **B)** $\mathbf{p}' \succ \mathbf{p}$ &emsp; **C)** $\mathbf{p} \succeq \mathbf{p}'$ and $\mathbf{p}' \succeq \mathbf{p}$ &emsp; **D)** $\mathbf{p} \succ \mathbf{p}'$

In [425]:
def majorizes(p, q):
    """Check if p majorizes q (p ≻ q): all partial sums of p↓ >= q↓"""
    p_sorted = sorted(p, reverse=True)
    q_sorted = sorted(q, reverse=True)
    p_cumsum = [sum(p_sorted[:k+1]) for k in range(len(p_sorted))]
    q_cumsum = [sum(q_sorted[:k+1]) for k in range(len(q_sorted))]
    return all(ps >= qs - 1e-10 for ps, qs in zip(p_cumsum, q_cumsum))

In [426]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
p_ = [16/61 *2, 11/61, 10/61, 10/61, 6/61, 8/61]
p_prime = [x / sum(p_) for x in p_]

In [427]:
p = p
q = p_prime

p_maj_q = majorizes(p, q)
q_maj_p = majorizes(q, p)

if p_maj_q and q_maj_p:
    print("p = q (mutual majorization)")
elif p_maj_q:
    print("p ≻ q (p majorizes q)")
elif q_maj_p:
    print("q ≻ p (q majorizes p)")
else:
    print("Incomparable (neither majorizes the other)")

q ≻ p (q majorizes p)


#### Problem 9 - just like 3, shannon entropy
Streaming the sequence of winners, how many bits do we now need on average per race?

   **A)** 1.623 &emsp; **B)** 1.607 &emsp; **C)** 0.1986 &emsp; **D)** 2.31
Answer D

In [428]:
H = -sum(pi * math.log2(pi) for pi in p_prime if pi > 0)
print(f'H(X) = {H:.3f} bits')

H(X) = 2.319 bits


#### Problem 10- bit loss
If we do not switch encodings, how much bits do we waste per race?

   **A)** 0.07375 &emsp; **B)** 0.2759 &emsp; **C)** 0.2022 &emsp; **D)** 0.07954
Answer C

In [429]:
H = -sum(pi * math.log2(pi) for pi in p if pi > 0)
H_ = -sum(pi * math.log2(pi) for pi in p_prime if pi > 0)
print(f'H(X) = {H:.3f} bits')
print(f'H(X\') = {H_:.3f} bits')
print('bit loss due to betting pattern:', H - H_)

H(X) = 2.521 bits
H(X') = 2.319 bits
bit loss due to betting pattern: 0.20215957793816797


## Course Miscellaneous

### Expected Profit

In [430]:
amounts = [9000, 4500, 18000, 6000, 3000, 4500]
p = [0.1, 0.2, 0.05, 0.15, 0.3, 0.2]
ex = sum(ai * pi for ai, pi in zip(amounts, p))
print(f'Expected payout = ${ex:.2f}')

Expected payout = $5400.00


### Expected Code Length - Huffman Coding

In [431]:
p = [2/12, 1/12, 1/12, 2/12, 4/12, 2/12]
code = ["100", "1010", "1011", "110", "0", "111"]
e_ln = sum(len(xi) * pi for xi, pi in zip(code, p))
print(str(e_ln) + "n") # n letters with the distribution

2.5n


### Information content of a series of events

In [432]:
p = [2/12, 1/12, 1/12, 2/12, 4/12, 2/12]
print([-math.log2(pi) for pi in p if pi > 0])

[2.584962500721156, 3.584962500721156, 3.584962500721156, 2.584962500721156, 1.5849625007211563, 2.584962500721156]


### Cross entropy

In [433]:
p = [2/12, 1/12, 1/12, 2/12, 4/12, 2/12]
q = [0.25, 0.2, 0.2, 0.2, 0.1, 0.05]
cross_h = -sum(qi * math.log2(pi) for qi, pi in zip(q, p))
print("Cross entropy " + str(cross_h))

Cross entropy 2.8849625007211563


### Relative entropy

In [434]:
p = [2/12, 1/12, 1/12, 2/12, 4/12, 2/12]
q = [0.25, 0.2, 0.2, 0.2, 0.1, 0.05]
cross_h = -sum(qi * math.log2(pi) for qi, pi in zip(q, p))
h_y = -sum(pi * math.log2(pi) for pi in q if pi > 0)
print("Cross entropy " + str(cross_h))
print("Entropy of Y " + str(h_y))
print("Relative entropy = Cross entropy - Entropy of Y " + str(cross_h - h_y))

Cross entropy 2.8849625007211563
Entropy of Y 2.4414460711655215
Relative entropy = Cross entropy - Entropy of Y 0.44351642955563486


### Lorenz Curve

In [435]:
def lorenz(G):
    """Calculate S and L from a distribution matrix G (values in row 0, probabilities in row 1)."""
    # Sort by values (ascending)
    order = np.argsort(G[0])
    x = G[0][order]
    p = G[1][order]

    S = np.cumsum(x * p)
    L = S / S[-1]
    return S, L

In [436]:
G = np.array([[1, 2, 3, 4, 5, 6],
              [0.3, 0.2, 0.2, 0.15, 0.1, 0.05]])

S, L = lorenz(G)
print(S,L)

[0.3 0.7 1.3 1.9 2.4 2.7] [0.11111111 0.25925926 0.48148148 0.7037037  0.88888889 1.        ]


### Schur-Convexity

In [437]:
def majorizes(x, y):
    """Check if x majorizes y (x ≻ y)."""
    x_sorted = np.sort(x)[::-1]
    y_sorted = np.sort(y)[::-1]
    if not np.isclose(x_sorted.sum(), y_sorted.sum()):
        return False
    return np.all(np.cumsum(x_sorted) >= np.cumsum(y_sorted) - 1e-10)

def compare_majorization(x, y):
    """Return the majorization relationship between x and y."""
    x_maj_y = majorizes(x, y)
    y_maj_x = majorizes(y, x)
    if x_maj_y and y_maj_x:
        return "equivalent"  # x and y are in the same orbit (permutations)
    elif x_maj_y:
        return "x ≻ y"
    elif y_maj_x:
        return "y ≻ x"
    else:
        return "incomparable"

def check_schur_convexity(f, n=3, num_tests=10000, eps=1e-6):
    violations = []

    for _ in range(num_tests):
        x = np.sort(np.random.rand(n))[::-1]
        x = x / x.sum()  # normalize so sums match
        i, j = np.random.choice(n, 2, replace=False)
        t = np.random.rand()
        y = x.copy()
        y[i] = t * x[i] + (1 - t) * x[j]
        y[j] = (1 - t) * x[i] + t * x[j]

        rel = compare_majorization(x, y)
        fx, fy = f(x), f(y)

        if rel == "x ≻ y" and fx < fy - eps:
            violations.append((x.copy(), y.copy(), rel, fx, fy))
        elif rel == "y ≻ x" and fy < fx - eps:
            violations.append((x.copy(), y.copy(), rel, fx, fy))

    # Schur-Ostrowski check
    ostrowski_violations = []
    h = 1e-7
    for _ in range(num_tests):
        x = np.random.rand(n)
        grad = np.zeros(n)
        for k in range(n):
            e = np.zeros(n)
            e[k] = h
            grad[k] = (f(x + e) - f(x - e)) / (2 * h)
        for i in range(n):
            for j in range(i + 1, n):
                val = (x[i] - x[j]) * (grad[i] - grad[j])
                if val < -eps:
                    ostrowski_violations.append((x.copy(), i, j, val))

    is_convex = len(violations) == 0 and len(ostrowski_violations) == 0
    return {
        "majorization_violations": violations,
        "ostrowski_violations": ostrowski_violations,
        "is_schur_convex": is_convex,
    }

def check_pair(f, x, y):
    """Check Schur convexity for a specific pair, reporting the majorization relationship."""
    rel = compare_majorization(x, y)
    fx, fy = f(x), f(y)
    consistent = None

    if rel == "x ≻ y":
        consistent = fx >= fy - 1e-10
    elif rel == "y ≻ x":
        consistent = fy >= fx - 1e-10
    elif rel == "equivalent":
        consistent = np.isclose(fx, fy)
    else:
        consistent = None  # can't say anything

    return {
        "relation": rel,
        "f(x)": fx,
        "f(y)": fy,
        "consistent_with_schur_convexity": consistent,
    }

In [438]:
f = lambda x: np.sum(x**2)

# Comparable pair
print(check_pair(f, np.array([0.5, 0.3, 0.2]), np.array([0.34, 0.33, 0.33])))
# relation: "x ≻ y", f(x) >= f(y) → consistent: True

# Incomparable pair
print(check_pair(f, np.array([0.5, 0.4, 0.1]), np.array([0.6, 0.25, 0.15])))
# relation: "incomparable", consistent: None (no conclusion possible)

# Equivalent pair (permutation)
print(check_pair(f, np.array([0.3, 0.5, 0.2]), np.array([0.5, 0.3, 0.2])))
# relation: "equivalent", f(x) ≈ f(y) → consistent: True

{'relation': 'x ≻ y', 'f(x)': np.float64(0.38), 'f(y)': np.float64(0.33340000000000003), 'consistent_with_schur_convexity': np.True_}
{'relation': 'incomparable', 'f(x)': np.float64(0.42000000000000004), 'f(y)': np.float64(0.445), 'consistent_with_schur_convexity': None}
{'relation': 'equivalent', 'f(x)': np.float64(0.38), 'f(y)': np.float64(0.38), 'consistent_with_schur_convexity': np.True_}


## Complicated stuff

### A bunch of entropies

In [439]:
def h_phi_entropy(p, h, phi):
    """
    Compute the (h, φ)-entropy of a probability vector p.
    
    h:   outer function R -> R
    phi: inner function applied to each p_i
    """
    return h(sum(phi(pi) for pi in p))

# --- Special cases ---

def shannon_entropy(p):
    return h_phi_entropy(p,
        h=lambda x: -x,
        phi=lambda pi: pi * np.log2(pi) if pi > 0 else 0)

def renyi_entropy(p, alpha):
    assert alpha != 1, "Use Shannon entropy for alpha=1"
    return h_phi_entropy(p,
        h=lambda x: np.log2(x) / (1 - alpha),
        phi=lambda pi: pi**alpha)

def tsallis_entropy(p, alpha):
    assert alpha != 1, "Use Shannon entropy for alpha=1"
    return h_phi_entropy(p,
        h=lambda x: (x - 1) / (1 - alpha),
        phi=lambda pi: pi**alpha)

def min_entropy(p):
    return -np.log2(max(p))

def collision_entropy(p):
    """Rényi entropy of order 2."""
    return renyi_entropy(p, alpha=2)

In [440]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]

print(f"Shannon:    {shannon_entropy(p):.4f}")
print(f"Rényi α=2:  {renyi_entropy(p, 2):.4f}")
print(f"Tsallis α=2:{tsallis_entropy(p, 2):.4f}")
print(f"Min-entropy:{min_entropy(p):.4f}")

# Custom (h, φ)-entropy
custom = h_phi_entropy(p,
    h=lambda x: np.sqrt(x),
    phi=lambda pi: pi**3)
print(f"Custom:     {custom:.4f}")

Shannon:    2.5209
Rényi α=2:  2.4585
Tsallis α=2:0.8181
Min-entropy:1.9307
Custom:     0.1895


### And a bunch of divergences

In [441]:
def h_phi_divergence(p, q, h, phi):
    """
    Compute the (h, φ)-divergence between distributions p and q.
    
    D_{h,φ}(p || q) = h( Σ q_i · φ(p_i / q_i) )
    
    h:   outer function R -> R
    phi: inner function applied to each ratio p_i/q_i
    """
    return h(sum(qi * phi(pi / qi) for pi, qi in zip(p, q) if qi > 0))

# --- Special cases ---

def kl_divergence(p, q):
    """Kullback-Leibler divergence D_KL(p || q)."""
    return h_phi_divergence(p, q,
        h=lambda x: x,
        phi=lambda t: t * np.log2(t) if t > 0 else 0)

def renyi_divergence(p, q, alpha):
    """Rényi divergence of order α."""
    assert alpha != 1, "Use KL divergence for alpha=1"
    return h_phi_divergence(p, q,
        h=lambda x: np.log2(x) / (alpha - 1),
        phi=lambda t: t**alpha)

def tsallis_divergence(p, q, alpha):
    """Tsallis divergence of order α."""
    assert alpha != 1, "Use KL divergence for alpha=1"
    return h_phi_divergence(p, q,
        h=lambda x: (x - 1) / (alpha - 1),
        phi=lambda t: t**alpha)

def hellinger_divergence(p, q):
    """Squared Hellinger distance."""
    return h_phi_divergence(p, q,
        h=lambda x: x,
        phi=lambda t: (np.sqrt(t) - 1)**2)

def chi_squared_divergence(p, q):
    """Chi-squared divergence."""
    return h_phi_divergence(p, q,
        h=lambda x: x,
        phi=lambda t: (t - 1)**2)

def tv_divergence(p, q):
    """Total variation distance."""
    return h_phi_divergence(p, q,
        h=lambda x: x / 2,
        phi=lambda t: abs(t - 1))

In [442]:
p = [16/61, 11/61, 10/61, 10/61, 6/61, 8/61]
q = [1/6] * 6

print(f"KL(p||q):       {kl_divergence(p, q):.4f}")
print(f"Rényi α=2:      {renyi_divergence(p, q, 2):.4f}")
print(f"Tsallis α=2:    {tsallis_divergence(p, q, 2):.4f}")
print(f"Hellinger:      {hellinger_divergence(p, q):.4f}")
print(f"Chi-squared:    {chi_squared_divergence(p, q):.4f}")
print(f"Total variation:{tv_divergence(p, q):.4f}")

KL(p||q):       0.0641
Rényi α=2:      0.1265
Tsallis α=2:    0.0916
Hellinger:      0.0222
Chi-squared:    0.0916
Total variation:0.1093
